# Student Performance Tracker
**CODETECH Internship – Task 1**  
**Name:** Shaik Rehaman

This beginner-friendly project creates a sample student dataset, explores the data, trains a Linear Regression model, evaluates it, and predicts a student's expected final marks.

> The dataset in this notebook is synthetic (dummy data) for demonstration and learning. Replace it with an approved real dataset if your internship instructions require one.

## 1. Import the required libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Create folders for project files
os.makedirs("data", exist_ok=True)
os.makedirs("output", exist_ok=True)
os.makedirs("screenshots", exist_ok=True)

# Make results reproducible
np.random.seed(42)

print("Libraries imported successfully!")

## 2. Create the student dataset

In [ ]:
# Create sample data for 300 students
number_of_students = 300

study_hours = np.random.uniform(0.5, 10, number_of_students)
attendance = np.random.uniform(40, 100, number_of_students)
previous_marks = np.random.uniform(30, 100, number_of_students)

# Create final marks using a simple relationship plus small random variation
noise = np.random.normal(0, 4, number_of_students)
final_marks = (
    3.5 * study_hours
    + 0.20 * attendance
    + 0.35 * previous_marks
    + 9
    + noise
)
final_marks = np.clip(final_marks, 0, 100)

# Store all values in a DataFrame
df = pd.DataFrame({
    "Study_Hours": study_hours.round(1),
    "Attendance_Percentage": attendance.round(1),
    "Previous_Marks": previous_marks.round(1),
    "Final_Marks": final_marks.round(1)
})

# Save the dataset
df.to_csv("data/student_data.csv", index=False)

print("Dataset created successfully!")
display(df.head())
print("Dataset shape:", df.shape)

## 3. Clean and understand the data

In [ ]:
# Check missing values
print("Missing values in each column:")
print(df.isnull().sum())

# Remove duplicate rows, if any
df = df.drop_duplicates().copy()

# Fill any missing numeric values with the column median
for column in df.select_dtypes(include="number").columns:
    df[column] = df[column].fillna(df[column].median())

print("\nDataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe())

## 4. Explore and visualize student performance

In [ ]:
# Distribution of final marks
plt.figure(figsize=(8, 5))
sns.histplot(df["Final_Marks"], bins=20, kde=True)
plt.title("Distribution of Students' Final Marks")
plt.xlabel("Final Marks")
plt.ylabel("Number of Students")
plt.tight_layout()
plt.savefig("output/final_marks_distribution.png", dpi=150)
plt.show()

# Relationship between study hours and final marks
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Study_Hours", y="Final_Marks")
plt.title("Study Hours vs Final Marks")
plt.tight_layout()
plt.savefig("output/study_hours_vs_marks.png", dpi=150)
plt.show()

# Correlation heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="Blues", fmt=".2f")
plt.title("Student Data Correlation")
plt.tight_layout()
plt.savefig("output/correlation_heatmap.png", dpi=150)
plt.show()

## 5. Train the Machine Learning model

In [ ]:
# Input features used to make predictions
X = df[["Study_Hours", "Attendance_Percentage", "Previous_Marks"]]

# Value the model learns to predict
y = df["Final_Marks"]

# Split data: 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Create and train a Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model training completed!")

## 6. Evaluate the model

In [ ]:
# Predict marks for the test dataset
y_pred = model.predict(X_test)

# Calculate evaluation scores
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared Score: {r2:.3f}")

# Save actual and predicted values
evaluation_results = pd.DataFrame({
    "Actual_Marks": y_test.values,
    "Predicted_Marks": y_pred.round(2)
})
evaluation_results.to_csv("output/model_evaluation_results.csv", index=False)

# Plot actual vs predicted marks
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel("Actual Marks")
plt.ylabel("Predicted Marks")
plt.title("Actual vs Predicted Student Marks")
plt.tight_layout()
plt.savefig("output/actual_vs_predicted.png", dpi=150)
plt.show()

## 7. Predict marks for a new student

In [ ]:
def get_performance_category(marks):
    """Return a simple category based on predicted marks."""
    if marks < 50:
        return "Needs Improvement"
    elif marks < 70:
        return "Average"
    elif marks < 85:
        return "Good"
    else:
        return "Excellent"

# Example student details (you can change these values)
new_student = pd.DataFrame({
    "Study_Hours": [6],
    "Attendance_Percentage": [85],
    "Previous_Marks": [75]
})

predicted_marks = float(model.predict(new_student)[0])
predicted_marks = float(np.clip(predicted_marks, 0, 100))
category = get_performance_category(predicted_marks)

print("STUDENT PERFORMANCE TRACKER")
print("----------------------------")
print("Study Hours:", new_student.loc[0, "Study_Hours"])
print("Attendance:", new_student.loc[0, "Attendance_Percentage"], "%")
print("Previous Marks:", new_student.loc[0, "Previous_Marks"])
print(f"Predicted Final Marks: {predicted_marks:.2f}")
print("Performance Category:", category)

# Save the example prediction
prediction_result = new_student.copy()
prediction_result["Predicted_Final_Marks"] = round(predicted_marks, 2)
prediction_result["Performance_Category"] = category
prediction_result.to_csv("output/prediction_results.csv", index=False)

## 8. Save the trained model

In [ ]:
import pickle

with open("output/student_performance_model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Trained model saved to output/student_performance_model.pkl")

## 9. Final project files

In [ ]:
print("Files created:")
for folder in ["data", "output"]:
    print(f"\\n{folder}/")
    for filename in sorted(os.listdir(folder)):
        print(" -", filename)